# **FINAL MODEL EVALUATION**

In [1]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
"""
Test Set Evaluation
"""

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score
)
import numpy as np
from datetime import datetime

# ============================================================================
# CONFIGURATION
# ============================================================================

# Paths
MODEL_CHECKPOINT = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/models/s4_50epochs/checkpoint-1150"
TEST_DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/data/processed/cycle3/test_manual_500.parquet"
RESULTS_DIR = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/results/"

# Model config
MAX_LENGTH = 512
BATCH_SIZE = 16

# ============================================================================
# LOAD MODEL AND DATA
# ============================================================================

print("="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)
print(f"Timestamp: {datetime.now()}")
print(f"Model: {MODEL_CHECKPOINT}")

# Load test data
test_df = pd.read_parquet(TEST_DATA_PATH)
print(f"Test set size: {len(test_df)}")
print(f"Positive examples: {test_df['has_claim'].sum()} ({test_df['has_claim'].mean():.1%})\n")

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Maltehb/danish-bert-botxo")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)
model.eval()
print("Model loaded\n")


FINAL TEST SET EVALUATION
Timestamp: 2026-05-24 10:14:40.487851
Model: /content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/models/s4_50epochs/checkpoint-1150
Test set size: 500
Positive examples: 49 (9.8%)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded



In [4]:
# ============================================================================
# INFERENCE
# ============================================================================
predictions = []
probabilities = []

with torch.no_grad():
    for i in range(0, len(test_df), BATCH_SIZE):
        batch = test_df.iloc[i:i+BATCH_SIZE]

        # Tokenize
        inputs = tokenizer(
            batch['ArticleText'].tolist(),
            max_length=MAX_LENGTH,
            truncation=True,
            padding=True,
            return_tensors='pt'
        )

        # Predict
        outputs = model(**inputs)
        logits = outputs.logits

        # Get predictions
        batch_probs = torch.softmax(logits, dim=1)[:, 1].numpy()
        batch_preds = (batch_probs > 0.5).astype(int)

        predictions.extend(batch_preds)
        probabilities.extend(batch_probs)

        if (i // BATCH_SIZE) % 5 == 0:
            print(f"  Processed {min(i+BATCH_SIZE, len(test_df))}/{len(test_df)}")

print("Inference complete\n")

# Add to dataframe
test_df['predicted'] = predictions
test_df['probability'] = probabilities


  Processed 16/500
  Processed 96/500
  Processed 176/500
  Processed 256/500
  Processed 336/500
  Processed 416/500
  Processed 496/500
Inference complete



In [7]:
# ============================================================================
# METRICS
# ============================================================================

y_true = test_df['has_claim'].values
y_pred = test_df['predicted'].values

print("="*60)
print("RESULTS")
print("="*60)

# Main metrics
f1 = f1_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
balanced_acc = balanced_accuracy_score(y_true, y_pred)

print(f"F1 Score:          {f1:.4f}")
print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(f"  True Negatives:  {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  True Positives:  {tp}")
print()

# Detailed report
print("Classification Report:")
print(classification_report(y_true, y_pred,
                          target_names=['No claim', 'Has claim'],
                          digits=4))

# ============================================================================
# ERROR ANALYSIS
# ============================================================================

print("="*60)
print("ERROR ANALYSIS")
print("="*60)

# False Positives (predicted 1, actually 0)
fp_df = test_df[(test_df['predicted'] == 1) & (test_df['has_claim'] == 0)]
print(f"\nFalse Positives: {len(fp_df)}")
if len(fp_df) > 0:
    print("\nSample False Positives (predicted claim, but no claim):")
    for idx, row in fp_df.nlargest(3, 'probability').iterrows():
        print(f"\n  Probability: {row['probability']:.3f}")
        print(f"  Text: {row['ArticleText'][:300]}...")

# False Negatives (predicted 0, actually 1)
fn_df = test_df[(test_df['predicted'] == 0) & (test_df['has_claim'] == 1)]
print(f"\nFalse Negatives: {len(fn_df)}")
if len(fn_df) > 0:
    print("\nSample False Negatives (missed claims):")
    for idx, row in fn_df.nsmallest(3, 'probability').iterrows():
        print(f"\n  Probability: {row['probability']:.3f}")
        print(f"  Text: {row['ArticleText'][:300]}...")

RESULTS
F1 Score:          0.2927
Precision:         0.3636
Recall:            0.2449
Balanced Accuracy: 0.5992

Confusion Matrix:
  True Negatives:  430
  False Positives: 21
  False Negatives: 37
  True Positives:  12

Classification Report:
              precision    recall  f1-score   support

    No claim     0.9208    0.9534    0.9368       451
   Has claim     0.3636    0.2449    0.2927        49

    accuracy                         0.8840       500
   macro avg     0.6422    0.5992    0.6148       500
weighted avg     0.8662    0.8840    0.8737       500

ERROR ANALYSIS

False Positives: 21

Sample False Positives (predicted claim, but no claim):

  Probability: 0.999
  Text: Lastbilchaufførernes demonstration bliver langt fra de sidste eksempler på, at de danske klimaambitioner altså kommer med en pris, mener chefredaktør på LandbrugsAvisen Chrisitan Friis Hansen.  I denne uge oplevede bilister flere steder i landet, at det kneb med at komme frem, fordi lastbiler enten ...

 

In [8]:
# For test predictions
print("Probability distribution:")
print(f"Predicted positive (mean prob): {test_df[test_df['predicted']==1]['probability'].mean():.3f}")
print(f"Predicted negative (mean prob): {test_df[test_df['predicted']==0]['probability'].mean():.3f}")

# Are predictions confident or borderline?
print(f"\nHigh confidence positive (prob > 0.7): {(test_df['probability'] > 0.7).sum()}")
print(f"Borderline (0.4 < prob < 0.6): {((test_df['probability'] > 0.4) & (test_df['probability'] < 0.6)).sum()}")
print(f"High confidence negative (prob < 0.3): {(test_df['probability'] < 0.3).sum()}")

Probability distribution:
Predicted positive (mean prob): 0.981
Predicted negative (mean prob): 0.001

High confidence positive (prob > 0.7): 33
Borderline (0.4 < prob < 0.6): 0
High confidence negative (prob < 0.3): 467


In [11]:
# False negatives
fn_df = test_df[(test_df['predicted'] == 0) & (test_df['has_claim'] == 1)]
print(f"False negative probabilities:")
print(fn_df['probability'].describe())

# True positives
tp_df = test_df[(test_df['predicted'] == 1) & (test_df['has_claim'] == 1)]
print(f"True positive probabilities:")
print(tp_df['probability'].describe())

False negative probabilities:
count    37.000000
mean      0.000947
std       0.003368
min       0.000022
25%       0.000025
50%       0.000029
75%       0.000075
max       0.019675
Name: probability, dtype: float64
True positive probabilities:
count    12.000000
mean      0.983876
std       0.040854
min       0.855722
25%       0.995731
50%       0.997664
75%       0.998252
max       0.999248
Name: probability, dtype: float64


In [13]:
# Which types of claims did the model catch vs miss?
print("True Positives - Superclaim breakdown:")
print(tp_df['HumanClaim'].value_counts())

print("\nFalse Negatives - Superclaim breakdown:")
print(fn_df['HumanClaim'].value_counts())

True Positives - Superclaim breakdown:
HumanClaim
[4]       9
[5, 6]    1
[4, 5]    1
[4, 6]    1
Name: count, dtype: int64

False Negatives - Superclaim breakdown:
HumanClaim
[4]          26
[6]           2
[4, 7]        1
[5]           1
[1]           1
[4, 6, 7]     1
[7]           1
[2]           1
[4, 6]        1
[4, 5, 7]     1
[3, 7]        1
Name: count, dtype: int64
